In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
using StaticArrays
using FastChebInterp

include("Chebyshev.jl")
using .Chebyshev

In [3]:
struct TransformedChebyshevSeries{T, N, F<:Function, G<:Function, H<:Function}
    series::ChebyshevSeries{T, N}
    u::F
    ∇u::G
    Hu::H

    function TransformedChebyshevSeries(
        series::ChebyshevSeries{T, N}, u::F, ∇u::G, Hu::H,
    ) where {T, N, F<:Function, G<:Function, H<:Function}
        
        if !all(hasmethod.([u, ∇u, Hu], Tuple{SVector{N, T}}))
            error("The transformation function must accept an argument of type SVector{$N, $T}")
        end

        u_return_type = Core.Compiler.return_type(u, Tuple{SVector{N, T}})
        u_proper_type = SVector{N, T}
        if !(u_return_type <: u_proper_type)
            error("The transformation function must return a $u_proper_type")
        end
        
        ∇u_return_type = Core.Compiler.return_type(∇u, Tuple{SVector{N, T}})
        ∇u_proper_type = SMatrix{N, N, T, N^2}
        if !(∇u_return_type <: ∇u_proper_type)
            error("The gradient of the transformation function must return a $∇u_proper_type")
        end

        Hu_return_type = Core.Compiler.return_type(Hu, Tuple{SVector{N, T}})
        Hu_proper_type = SArray{Tuple{N, N, N}, T, 3, N^3}
        if !(Hu_return_type <: Hu_proper_type)
            error("The hessian of the transformation function must return a $Hu_proper_type")
        end

        return new{T, N, F, G, H}(series, u, ∇u, Hu)
    end
end;

In [4]:
function f(u)
    x = u.^2
    return sin.(x)
end

function ∇f(u)
    x = u.^2
    return cos.(x)
end

function Hf(u)
    x = u.^2
    return -sin.(x)
end

function u(x::SVector{1, Float64})
    return x.^0.5
end

function ∇u(x::SVector{1, Float64})
    return reshape(0.5*x.^-0.5, Size(1, 1))
end

function Hu(x::SVector{1, Float64})
    return reshape(-0.25*x.^-1.5, Size(1, 1, 1))
end

# function u(x::Float64)
#     return x.^0.5
# end

# function ∇u(x::Float64)
#     return 0.5*x.^-0.5
# end

# function Hu(x::Float64)
#     return -0.25*x.^-1.5
# end

x1, x2 = SA[0.0], SA[0.5*π]
u1, u2 = u(x1), u(x2)

n = 50

xc = chebpoints(n, u1[], u2[])
cb = chebinterp(f.(xc), u1[], u2[])

cs = ChebyshevSeries(cb.coefs, u1[], u2[])
tcs = TransformedChebyshevSeries(cs, u, ∇u, Hu);

LoadError: expected tuple type

In [ ]:
tcs.series(0.5 * (u1[] + u2[]))

In [6]:
function fx(x::SVector{2, Float64})
    return exp(x[1]) * cos(x[2])
end

function f(u::SVector{2, Float64})
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return exp(x) * cos(y)
end

function ∇f(u)
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return SA[exp(x)*cos(y), -exp(x)*sin(y)]
end

function Hf(u)
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return SA[ exp(x)*cos(y) -exp(x)*sin(y);
              -exp(x)*sin(y) -exp(x)*cos(y)]
end

function u(x::SVector{2, Float64})
    return SA[sqrt(x[1]^2 + x[2]^2), atan(x[2]/x[1])]
end

function ∇u(x::SVector{2, Float64})
    r² = x[1]^2 + x[2]^2
    r = √r²
    
    ∇u₁ = SA[ x[1]/r , x[2]/r ]
    ∇u₂ = SA[-x[2]/r², x[1]/r²]
    
    return vcat(∇u₁', ∇u₂')
end

function Hu(x::SVector{2, Float64})
    xy = x[1]*x[2]
    x², y² = x.^2
    r² = x² + y²
    r = √r²
    r³ = r²*r
    r⁴ = r³*r

    Hu₁ = SA[y²/r³  -xy/r³;
             -xy/r³  x²/r³]
    Hu₂ = SA[2xy/r⁴     (y²-x²)/r⁴;
             (y²-x²)/r⁴ -2xy/r⁴   ]

    hess_u = zeros(SArray{Tuple{2, 2, 2}, Float64})
    hess_u[1, :, :] = Hu₁
    hess_u[2, :, :] = Hu₂
    
    return hess_u
end

function u_to_x(u::SVector{2, Float64})
    return SA[u[1]*cos(u[2]), u[1]*sin(u[2])]
end

u1 = SA[0.0, -0.2*π]
u2 = SA[2.0, 0.4*π]

n = (50, 50)

xc = chebpoints(n, u1, u2)
cb = chebinterp(f.(xc), u1, u2)

tf = Transformation(u, ∇u, Hu)
cs = ChebyshevSeries(cb.coefs, u1, u2)

tcs = TransformedChebyshevSeries(cs, u, ∇u, Hu);

LoadError: The gradient of the transformation function must return a SMatrix{2, 2, Float64, 4}

In [8]:
u3 = SA[0.6, -0.1]
x3 = u_to_x(u3);

tcs.series(u3)

1.8134070379886973

In [9]:
rtype = SArray{Tuple{1, 1, 1}, Float64}

a = 1

rtype(a)

1×1×1 SArray{Tuple{1, 1, 1}, Float64, 3, 1} with indices SOneTo(1)×SOneTo(1)×SOneTo(1):
[:, :, 1] =
 1.0

In [8]:
rtype = SVector{1, Float64}
rtype(1.0)

1-element SVector{1, Float64} with indices SOneTo(1):
 1.0

In [3]:
function f(u)
    x = u.^2
    return sin.(x)
end

function ∇f(u)
    x = u.^2
    return cos.(x)
end

function Hf(u)
    x = u.^2
    return -sin.(x)
end

function u(x::SVector{1, Float64})
    return x.^0.5
end

function ∇u(x::SVector{1, Float64})
    return reshape(0.5*x.^-0.5, 1, 1)
end

function Hu(x::SVector{1, Float64})
    return reshape(-0.25*x.^-1.5, 1, 1, 1)
end

# function u(x::Float64)
#     return x^0.5
# end

# function ∇u(x::Float64)
#     return 0.5*x^-0.5
# end

# function Hu(x::Float64)
#     return -0.25*x^-1.5
# end

x1, x2 = SA[0.0], SA[0.5*π]
u1, u2 = u(x1), u(x2)

n = 50

xc = chebpoints(n, u1[], u2[])
cb = chebinterp(f.(xc), u1[], u2[])

tf = Transformation(u, ∇u, Hu)
cs = ChebyshevSeries(cb.coefs, u1[], u2[])

cc = ChebyshevCluster((cs,), (tf,));

In [4]:
x3 = 0.5*(x1 + x2)
u3 = u(x3);

In [5]:
cc(x3[]) - f(u3)[]

1.1102230246251565e-16

In [6]:
g, ∇g = gradient(cc, x3[])

println(g, ", ", f(u3)[])
println(∇g, ", ", ∇f(u3)[])

println(g - f(u3)[])
println(∇g - ∇f(u3)[])

0.7071067811865477, 0.7071067811865476
0.7071067811865487, 0.7071067811865475
1.1102230246251565e-16
1.2212453270876722e-15


In [7]:
g, ∇g, Hg = hessian(cc, x3[])

println(g, ", ", f(u3)[])
println(∇g, ", ", ∇f(u3)[])
println(Hg, ", ", Hf(u3)[])

println(g - f(u3)[])
println(∇g - ∇f(u3)[])
println(Hg - Hf(u3)[])

0.7071067811865477, 0.7071067811865476
0.7071067811865487, 0.7071067811865475
-0.7071067811865537, -0.7071067811865476
1.1102230246251565e-16
1.2212453270876722e-15
-6.106226635438361e-15


In [8]:
function fx(x::SVector{2, Float64})
    return exp(x[1]) * cos(x[2])
end

function f(u::SVector{2, Float64})
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return exp(x) * cos(y)
end

function ∇f(u)
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return SA[exp(x)*cos(y), -exp(x)*sin(y)]
end

function Hf(u)
    x = u[1]*cos(u[2])
    y = u[1]*sin(u[2])
    return SA[ exp(x)*cos(y) -exp(x)*sin(y);
              -exp(x)*sin(y) -exp(x)*cos(y)]
end

function u(x::SVector{2, Float64})
    return SA[sqrt(x[1]^2 + x[2]^2), atan(x[2]/x[1])]
end

function ∇u(x::SVector{2, Float64})
    r² = x[1]^2 + x[2]^2
    r = √r²
    
    ∇u₁ = SA[ x[1]/r , x[2]/r ]
    ∇u₂ = SA[-x[2]/r², x[1]/r²]
    
    return vcat(∇u₁', ∇u₂')
end

function Hu(x::SVector{2, Float64})
    xy = x[1]*x[2]
    x², y² = x.^2
    r² = x² + y²
    r = √r²
    r³ = r²*r
    r⁴ = r³*r

    Hu₁ = SA[y²/r³  -xy/r³;
             -xy/r³  x²/r³]
    Hu₂ = SA[2xy/r⁴     (y²-x²)/r⁴;
             (y²-x²)/r⁴ -2xy/r⁴   ]

    hess_u = Array{Float64, 3}(undef, 2, 2, 2)
    hess_u[1, :, :] = Hu₁
    hess_u[2, :, :] = Hu₂
    
    return hess_u
end

function u_to_x(u::SVector{2, Float64})
    return SA[u[1]*cos(u[2]), u[1]*sin(u[2])]
end

u1 = SA[0.0, -0.2*π]
u2 = SA[2.0, 0.4*π]

n = (50, 50)

xc = chebpoints(n, u1, u2)
cb = chebinterp(f.(xc), u1, u2)

tf = Transformation(u, ∇u, Hu)
cs = ChebyshevSeries(cb.coefs, u1, u2)

cc = ChebyshevCluster((cs,), (tf,));

In [9]:
# u3 = u1 + 0.3*(u2 - u1)
u3 = SA[0.6, -0.1]
x3 = u_to_x(u3);

In [10]:
g = cc(x3)

println(g)
println(f(u3))
println()

cc(x3) - f(u3)

1.8134070379886973
1.813407037988697



2.220446049250313e-16

In [11]:
g, ∇g = gradient(cc, x3)

println(g)
println(f(u3))
println()
println(∇g)
println(∇f(u3))
println()

println(g - f(u3))
println(∇g - ∇f(u3))

1.8134070379886973
1.813407037988697

[1.8134070379887013, 0.10875327284161514]
[1.813407037988697, 0.1087532728416087]

2.220446049250313e-16
[4.218847493575595e-15, 6.439293542825908e-15]


In [12]:
g, ∇g, Hg = hessian(cc, x3)

println(g)
println(f(u3))
println()
println(∇g)
println(∇f(u3))
println()
println(Hg)
println(Hf(u3))
println()

println(g - f(u3))
println(∇g - ∇f(u3))
println(Hg - Hf(u3))

1.8134070379886973
1.813407037988697

[1.8134070379887013, 0.10875327284161514]
[1.813407037988697, 0.1087532728416087]

[1.8134070379886607 0.10875327284156461; 0.10875327284156455 -1.8134070379894585]
[1.813407037988697 0.1087532728416087; 0.1087532728416087 -1.813407037988697]

2.220446049250313e-16
[4.218847493575595e-15, 6.439293542825908e-15]
[-3.6415315207705135e-14 -4.408973186542653e-14; -4.414524301665779e-14 -7.613909502879324e-13]
